# 📖 Notebook 2: Activity Feed & Social Features

Strava isn't just about tracking your own runs — it's about seeing what your friends are doing.
In this notebook we build the **social feed**: querying friends' activities, pagination,
and caching feeds in Redis to avoid hammering the database.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **bi-directional friendships** work in a database
- How to query a **friends activity feed** with pagination
- Why feed queries get expensive at scale
- How to cache feeds in **Redis** for fast reads
- The difference between **fan-out-on-write** and **fan-out-on-read**

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/strava
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `strava_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "strava_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 👥 Friendships: A Bi-Directional Graph

In Strava, friendships are **mutual** — if Alice follows Bob, Bob also follows Alice.

In the database, we store this with **two rows** per friendship:

```
friends table:
  user_id=1, friend_id=2   ← Alice → Bob
  user_id=2, friend_id=1   ← Bob → Alice
```

This makes queries simple: "give me all friends of user X" is just
`SELECT friend_id FROM friends WHERE user_id = X`.

The trade-off is we store 2× the rows, but friendship tables are small.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Who are Alice's friends?
cur.execute("""
    SELECT u.id, u.username, u.display_name, u.city
    FROM friends f
    JOIN users u ON f.friend_id = u.id
    WHERE f.user_id = 1
    ORDER BY u.username;
""")
friends = cur.fetchall()

print(f"Alice's friends ({len(friends)} total):")
print(f"{'ID':>4} {'Username':<10} {'Name':<20} {'City':<15}")
print("-" * 55)
for f in friends:
    print(f"{f['id']:>4} {f['username']:<10} {f['display_name']:<20} {f['city']:<15}")

conn.close()

## 📰 The Activity Feed: Two Modes

Strava's feed has two views:

1. **My Activities** (`mode=USER`) — your own completed runs and rides
2. **Friends Feed** (`mode=FRIENDS`) — completed activities from all your friends

Let's implement both with pagination.

In [ ]:
def get_activity_feed(user_id, mode='USER', page=1, page_size=5):
    """
    Fetch a page of activities.
    
    mode='USER'    → the user's own activities
    mode='FRIENDS' → activities from the user's friends
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    offset = (page - 1) * page_size
    
    if mode == 'USER':
        cur.execute("""
            SELECT a.id, u.username, a.type, a.title,
                   ROUND(a.distance_m::numeric) as distance_m,
                   a.duration_s, a.completed_at
            FROM activities a
            JOIN users u ON a.user_id = u.id
            WHERE a.state = 'COMPLETE'
              AND a.user_id = %s
            ORDER BY a.completed_at DESC
            LIMIT %s OFFSET %s;
        """, (user_id, page_size, offset))
    else:
        # Friends feed: activities from all friends
        cur.execute("""
            SELECT a.id, u.username, a.type, a.title,
                   ROUND(a.distance_m::numeric) as distance_m,
                   a.duration_s, a.completed_at
            FROM activities a
            JOIN users u ON a.user_id = u.id
            WHERE a.state = 'COMPLETE'
              AND a.user_id IN (
                  SELECT friend_id FROM friends WHERE user_id = %s
              )
            ORDER BY a.completed_at DESC
            LIMIT %s OFFSET %s;
        """, (user_id, page_size, offset))
    
    results = cur.fetchall()
    conn.close()
    return results

# Alice's own activities
print("=" * 70)
print("Alice's Activities (mode=USER)")
print("=" * 70)
my_feed = get_activity_feed(user_id=1, mode='USER', page=1)
for a in my_feed:
    print(f"  #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
          f"{a['distance_m']:>7} m  {a['duration_s']:>5} s  {a['title']}")

print()
print("=" * 70)
print("Alice's Friends Feed (mode=FRIENDS)")
print("=" * 70)
friends_feed = get_activity_feed(user_id=1, mode='FRIENDS', page=1)
for a in friends_feed:
    print(f"  #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
          f"{a['distance_m']:>7} m  {a['duration_s']:>5} s  {a['title']}")

In [ ]:
# Pagination demo — page through friends' activities
print("Paginating Alice's friends feed:")
print()

for page in range(1, 4):
    results = get_activity_feed(user_id=1, mode='FRIENDS', page=page, page_size=3)
    if not results:
        print(f"  Page {page}: (empty — no more activities)")
        break
    print(f"  Page {page}:")
    for a in results:
        print(f"    #{a['id']:>3} {a['username']:<8} {a['type']:<5} "
              f"{a['distance_m']:>7} m  {a['title']}")
    print()

print("💡 Pagination uses LIMIT/OFFSET. At scale, cursor-based pagination")
print("   (WHERE completed_at < last_seen_timestamp) performs better.")

## 📜 Bad → Best: Cursor-Based Pagination

We used `LIMIT / OFFSET` for pagination above. It works, but it has a sneaky
problem: **OFFSET forces the database to scan and discard rows**.

- Page 1 (OFFSET 0):   read 10 rows → return 10 ✅
- Page 10 (OFFSET 90): read 100 rows, **throw away 90**, return 10 😬
- Page 1000:           read 10,010 rows, throw away 10,000 😱

At Strava scale (millions of activities), this is unusable.

**The fix: keyset / cursor pagination.**  
Instead of "skip the first N", we say "give me rows *after* this timestamp".
The database jumps directly to that point using an index — O(log N) regardless
of the page number.

```sql
-- Bad:  read+discard N rows
LIMIT 10 OFFSET 990

-- Better: jump straight there via the index on completed_at
WHERE completed_at < :last_seen_at
ORDER BY completed_at DESC
LIMIT 10
```

### The trap: `completed_at` is not unique

A cursor has to identify **one row**, not a set of them. `completed_at` doesn't:
a watch syncing a backlog, a bulk import, or two rides finishing in the same
millisecond all produce ties. `WHERE completed_at < :cursor` then skips *every*
row that shares the cursor's timestamp — silently, with no error, and only for
users unlucky enough to have a tie land on a page boundary.

The fix is to make the cursor unique by appending the primary key and comparing
the pair. Postgres compares row values left to right, so this is still one index
range scan:

```sql
-- Best: a cursor that identifies exactly one row
WHERE (completed_at, id) < (:last_seen_at, :last_seen_id)
ORDER BY completed_at DESC, id DESC
LIMIT 10
```

Below we manufacture the tie, watch the naive cursor lose rows, then fix it.

In [ ]:
# Set up the tie: Bob's watch syncs a backlog of 5 rides that all land with the
# *same* completed_at. This is not exotic -- any bulk import produces it.
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM activities WHERE title LIKE 'Backlog sync %'")
cur.execute("""
    INSERT INTO activities (user_id, type, state, title, distance_m, duration_s,
                            started_at, completed_at)
    SELECT 2, 'RIDE', 'COMPLETE', 'Backlog sync #' || i, 5000 + i * 100, 1200,
           NOW() - INTERVAL '1 hour', NOW()
    FROM generate_series(1, 5) AS i;
""")
conn.close()
print("Seeded 5 of Bob's activities with an identical completed_at.\n")


def _fetch_page(where_sql, params, page_size):
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(f"""
        SELECT a.id, u.username, a.title, a.completed_at
        FROM activities a JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
          AND a.user_id IN (SELECT friend_id FROM friends WHERE user_id = %s)
          {where_sql}
        ORDER BY a.completed_at DESC, a.id DESC
        LIMIT %s;
    """, params + (page_size,))
    rows = cur.fetchall()
    conn.close()
    return rows


def feed_cursor_naive(user_id, cursor=None, page_size=3):
    """Bad: the cursor is a timestamp, so ties on completed_at get skipped."""
    if cursor is None:
        rows = _fetch_page("", (user_id,), page_size)
    else:
        rows = _fetch_page("AND a.completed_at < %s", (user_id, cursor), page_size)
    return rows, (rows[-1]["completed_at"] if rows else None)


def feed_cursor_keyset(user_id, cursor=None, page_size=3):
    """Best: the cursor is (completed_at, id) -- unique, so nothing is skipped."""
    if cursor is None:
        rows = _fetch_page("", (user_id,), page_size)
    else:
        ts, last_id = cursor
        rows = _fetch_page("AND (a.completed_at, a.id) < (%s, %s)",
                           (user_id, ts, last_id), page_size)
    return rows, ((rows[-1]["completed_at"], rows[-1]["id"]) if rows else None)


def page_through(fn, pages=4, page_size=3):
    """Walk `pages` pages and collect the activity ids we were shown."""
    seen, cursor = [], None
    for _ in range(pages):
        rows, cursor = fn(user_id=1, cursor=cursor, page_size=page_size)
        seen.extend(r["id"] for r in rows)
        if not rows:
            break
    return seen

naive_ids  = page_through(feed_cursor_naive)
keyset_ids = page_through(feed_cursor_keyset)

# Ground truth: the same query with no cursor at all, read in one shot.
truth_ids = [r["id"] for r in _fetch_page("", (1,), 12)]

print(f"Ground truth (one query, 12 rows): {truth_ids}")
print(f"Naive timestamp cursor:            {naive_ids}")
print(f"Keyset (completed_at, id) cursor:  {keyset_ids}")
print()
print(f"  Rows the naive cursor silently dropped: "
      f"{sorted(set(truth_ids) - set(naive_ids))}")
print(f"  Duplicates the naive cursor returned:   "
      f"{len(naive_ids) - len(set(naive_ids))}")

# The bug has to actually reproduce, or the "Best" version is teaching nothing.
lost = set(truth_ids) - set(naive_ids)
assert lost, (
    "the naive timestamp cursor did not lose any rows -- the tie we seeded is not "
    "landing on a page boundary, so this cell no longer demonstrates its own point"
)
# ...and the fix has to actually fix it: same rows, same order, no duplicates.
assert keyset_ids == truth_ids, (
    f"keyset pagination returned {keyset_ids}, expected {truth_ids}"
)
assert len(keyset_ids) == len(set(keyset_ids)), "keyset pagination returned duplicates"

print()
print(f"\U0001f4a1 The naive cursor lost {len(lost)} of Bob's backlog rides. Both queries")
print("   are index scans that don't grow with page depth — but only the keyset")
print("   version is *correct*. Ties are the reason cursors carry a primary key.")

## ⚡ The Problem: Feed Queries Are Expensive

Look at our friends feed query — it does:
1. A subquery to find all friend IDs
2. An `IN (...)` filter across all activities
3. A sort by completion time

With millions of users and activities, this gets slow fast.

Let's measure the cost.

In [ ]:
# Measure query time for the friends feed
conn = get_db()
cur = conn.cursor()

times_db = []
for _ in range(50):
    start = time.time()
    cur.execute("""
        SELECT a.id, u.username, a.type, a.title,
               a.distance_m, a.duration_s, a.completed_at
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
          AND a.user_id IN (
              SELECT friend_id FROM friends WHERE user_id = 1
          )
        ORDER BY a.completed_at DESC
        LIMIT 10;
    """)
    cur.fetchall()
    times_db.append((time.time() - start) * 1000)

conn.close()

avg_db = sum(times_db) / len(times_db)
print(f"Friends feed from PostgreSQL (50 queries):")
print(f"  Average: {avg_db:.2f} ms")
print(f"  Min:     {min(times_db):.2f} ms")
print(f"  Max:     {max(times_db):.2f} ms")
print()
print("💡 Seems fast with our small dataset.")
print("   But imagine 100M users with 500 friends each — this blows up.")
print("   That's where caching comes in.")

## 🚀 Caching Feeds in Redis

The idea: cache each user's friends feed in Redis so we skip the database entirely.

**Pattern: Cache-Aside**
1. Check Redis for the feed → if found, return it (**cache hit**)
2. If not in Redis → query Postgres → store result in Redis with a TTL
3. When a friend completes an activity → **invalidate** the cached feed

```
Redis key: feed:friends:{user_id}:page:{page}:size:{page_size}
Value:     JSON array of recent activities
TTL:       60 seconds (feeds are slightly stale, that's OK)
```

⚠️ **The cache key must contain every input that changes the answer.** The obvious
key here is `feed:friends:{user_id}`, but the query also depends on `page` *and*
`page_size` — leave either one out and a request for 3 rows happily gets served the
10-row payload cached by an earlier request. It is not a crash, it is wrong data, and
it only shows up once two clients use different page sizes.

In [ ]:
r = get_redis()

FEED_TTL = 60  # seconds

def feed_cache_key(user_id, page, page_size):
    """Every input that changes the result has to be in the key -- page_size included."""
    return f"feed:friends:{user_id}:page:{page}:size:{page_size}"


def get_friends_feed_cached(user_id, page=1, page_size=10):
    """
    Get friends feed with Redis caching.
    Returns (results, cache_hit: bool).
    """
    cache_key = feed_cache_key(user_id, page, page_size)

    # Step 1: Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # cache hit!

    # Step 2: Cache miss — query the database
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    offset = (page - 1) * page_size
    cur.execute("""
        SELECT a.id, u.username, a.type, a.title,
               ROUND(a.distance_m::numeric) as distance_m,
               a.duration_s
        FROM activities a
        JOIN users u ON a.user_id = u.id
        WHERE a.state = 'COMPLETE'
          AND a.user_id IN (
              SELECT friend_id FROM friends WHERE user_id = %s
          )
        ORDER BY a.completed_at DESC
        LIMIT %s OFFSET %s;
    """, (user_id, page_size, offset))
    results = cur.fetchall()
    conn.close()

    # Convert Decimal/datetime to JSON-friendly types
    serializable = []
    for row in results:
        serializable.append({
            'id': row['id'],
            'username': row['username'],
            'type': row['type'],
            'title': row['title'],
            'distance_m': float(row['distance_m']) if row['distance_m'] else 0,
            'duration_s': row['duration_s'],
        })

    # Step 3: Store in Redis with TTL
    r.setex(cache_key, FEED_TTL, json.dumps(serializable))

    return serializable, False  # cache miss

# First call: cache miss
feed, hit = get_friends_feed_cached(user_id=1)
print(f"Call 1: cache {'HIT ✅' if hit else 'MISS ❌'}  — {len(feed)} activities")

# Second call: cache hit!
feed, hit = get_friends_feed_cached(user_id=1)
print(f"Call 2: cache {'HIT ✅' if hit else 'MISS ❌'}  — {len(feed)} activities")

# Show the TTL
ttl = r.ttl(feed_cache_key(1, 1, 10))
print(f"\nTTL remaining: {ttl} seconds")

# A different page_size is a different query and must not reuse the cached payload.
# With the key `feed:friends:{user_id}:page:{page}` this returns 10 rows, not 3.
small, _ = get_friends_feed_cached(user_id=1, page=1, page_size=3)
assert len(small) == 3, (
    f"asked for a page of 3 and got {len(small)} rows back -- the cache key is "
    f"missing page_size and served the payload cached for another page size"
)
print(f"Same page, page_size=3: {len(small)} rows (a separate cache entry)")
print("\n💡 Open RedisInsight at http://localhost:5540 to see the cached keys!")

In [ ]:
# Measure the speed difference: cached vs uncached
r.delete(feed_cache_key(1, 1, 10))  # clear cache first

# Uncached (first call hits Postgres)
start = time.time()
for _ in range(50):
    r.delete(feed_cache_key(1, 1, 10))
    get_friends_feed_cached(user_id=1)
uncached_avg = ((time.time() - start) / 50) * 1000

# Cached (subsequent calls hit Redis)
get_friends_feed_cached(user_id=1)  # warm the cache
start = time.time()
for _ in range(50):
    get_friends_feed_cached(user_id=1)
cached_avg = ((time.time() - start) / 50) * 1000

print(f"Feed latency comparison (50 calls each):")
print(f"  Uncached (Postgres): {uncached_avg:.2f} ms")
print(f"  Cached (Redis):      {cached_avg:.2f} ms")
print(f"  Speedup:             {uncached_avg/cached_avg:.1f}×")

# The whole point of the cache is that it is faster. If it isn't, say so loudly.
assert cached_avg < uncached_avg, (
    f"the cache is not faster than Postgres ({cached_avg:.2f} ms vs "
    f"{uncached_avg:.2f} ms) -- with this dataset something is wrong"
)
print()
print("💡 Honest reading: most of this gap is one network round-trip plus query")
print("   planning, not data volume — our dataset is tiny. At millions of activities")
print("   the Postgres side grows and the Redis side does not; that's the real win.")

## 🔄 Cache Invalidation: When a Friend Completes an Activity

When Bob finishes a run, Alice's cached feed becomes **stale** — it doesn't include
Bob's new activity. We need to invalidate it.

**Strategy**: when any user completes an activity, delete all their friends' cached feeds.

```
Bob completes a run
  → Find all of Bob's friends: [Alice, Carol, Frank, ...]
  → Delete keys: feed:friends:1:*, feed:friends:3:*, feed:friends:6:*, ...
  → Next time Alice opens her feed, it rebuilds from Postgres
```

In [ ]:
def on_activity_completed(user_id):
    """
    Called when a user completes an activity.
    Invalidates all friends' cached feeds so they see the new activity.
    """
    conn = get_db()
    cur = conn.cursor()

    # Find all friends of this user
    cur.execute("SELECT friend_id FROM friends WHERE user_id = %s", (user_id,))
    friend_ids = [row[0] for row in cur.fetchall()]
    conn.close()

    # Delete each friend's cached feed
    invalidated = 0
    for fid in friend_ids:
        # Delete all pages of this friend's feed cache
        keys = r.keys(f"feed:friends:{fid}:page:*")   # every page, every page size
        if keys:
            r.delete(*keys)
            invalidated += len(keys)

    return friend_ids, invalidated

# Demo: Bob (user 2) completes a new activity
# First, make sure Alice's feed is cached
get_friends_feed_cached(user_id=1)
print(f"Alice's feed cached: {r.exists('feed:friends:1:page:1')} (1=yes)")

# Bob completes a run
friend_ids, count = on_activity_completed(user_id=2)
print(f"\nBob completed a run!")
print(f"  Friends notified: {friend_ids}")
print(f"  Cache keys invalidated: {count}")

# Alice's cache should be gone now
print(f"\nAlice's feed cached: {r.exists(feed_cache_key(1, 1, 10))} (0=no)")

# Invalidation that misses a page is worse than no cache at all: the user sees a
# feed that never updates. Check that nothing survived for any of Bob's friends.
survivors = [k for fid in friend_ids for k in r.keys(f"feed:friends:{fid}:page:*")]
assert not survivors, f"stale feed pages survived invalidation: {survivors}"
assert 1 in friend_ids, "Alice should be one of Bob's friends in the seed data"
print("\n💡 Next time Alice opens her feed, it will rebuild from Postgres")
print("   and include Bob's new activity.")

## 🤔 Fan-Out-on-Read vs Fan-Out-on-Write

There are two main ways to build a social feed:

### Fan-Out-on-Read (what we built above)
When Alice opens her feed, we query all her friends' activities on the spot.

```
Alice opens feed
  → Find Alice's friends
  → Query: SELECT ... WHERE user_id IN (friends) ORDER BY time
  → Return results
```

**Pros**: Simple, always fresh  
**Cons**: Slow at scale (many JOINs per request)

### Fan-Out-on-Write (Twitter/Instagram style)
When Bob completes a run, we immediately write it to all his friends' feeds.

```
Bob completes a run
  → Find Bob's friends: [Alice, Carol, Frank]
  → Append to each friend's feed list in Redis:
      LPUSH feed:alice {...}
      LPUSH feed:carol {...}
      LPUSH feed:frank {...}
```

**Pros**: Reads are instant (just read from pre-built list)  
**Cons**: Writes are expensive (celebrity with 10M followers = 10M writes)

### Which to Use?

| Scenario | Best Approach |
|----------|---------------|
| Most users have < 500 friends | Fan-out-on-write |
| Some users have millions of followers | Hybrid (fan-out-on-write for normal users, fan-out-on-read for celebrities) |
| Strava (fitness app, ~100-500 friends) | Either works — fan-out-on-read + cache is simpler |

For Strava, fan-out-on-read with caching (what we built) is a great fit because:
- Friend counts are small (hundreds, not millions)
- Activities happen at most a few times per day
- Slight staleness is acceptable

In [ ]:
# Let's also demonstrate fan-out-on-write to compare

def fanout_on_write(user_id, activity_data):
    """
    When a user completes an activity, push it to all friends' feed lists in Redis.
    """
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT friend_id FROM friends WHERE user_id = %s", (user_id,))
    friend_ids = [row[0] for row in cur.fetchall()]
    conn.close()

    payload = json.dumps(activity_data)
    for fid in friend_ids:
        r.lpush(f"feed:fanout:{fid}", payload)
        r.ltrim(f"feed:fanout:{fid}", 0, 49)  # keep only last 50

    return len(friend_ids)

def read_fanout_feed(user_id, count=5):
    """Read a pre-built feed from Redis (fan-out-on-write)."""
    items = r.lrange(f"feed:fanout:{user_id}", 0, count - 1)
    return [json.loads(item) for item in items]

# Bob completes a run → fan out to all friends
activity = {
    'id': 999, 'username': 'bob', 'type': 'RUN',
    'title': 'Evening Run', 'distance_m': 5200, 'duration_s': 1500
}
written_to = fanout_on_write(user_id=2, activity_data=activity)
print(f"Bob completed 'Evening Run' → fanned out to {written_to} friends")
print()

# Alice reads her pre-built feed
alice_feed = read_fanout_feed(user_id=1)
print("Alice's fan-out feed (from Redis list):")
for item in alice_feed:
    print(f"  {item['username']}: {item['title']} — {item['distance_m']} m")

assert any(item['id'] == 999 for item in alice_feed), (
    "Bob's activity is not at the top of Alice's fan-out feed -- the write fanned "
    "out to the wrong key or LPUSH/LTRIM dropped it"
)
assert r.llen("feed:fanout:1") <= 50, "LTRIM should cap each fan-out feed at 50 entries"
print()
print("💡 No database query needed! The feed was pre-built when Bob completed his run.")
print("   Note the cost side: this one activity caused "
      f"{written_to} writes. A celebrity with 10M")
print("   followers would cause 10M — which is why real systems go hybrid.")

## 📊 Real-Time Friend Tracking (Polling)

A common follow-up question: *"Can friends watch each other's runs in real-time?"*

The key insight from the source material: **polling is better than WebSockets here**.

Why?
- Updates arrive every 2–5 seconds (predictable interval)
- A few seconds of delay is perfectly acceptable
- Polling is simpler to implement and scale

```
Bob is running:
  Every 5s → phone sends GPS update to server → stored in Redis

Alice is watching:
  Every 7s → phone polls server → "Where is Bob now?" → Redis lookup
```

The two clocks are **independent** — nobody coordinates Bob's writes with Alice's
polls. So what Alice sees is always a little old, and the interesting question is
*how old*. The bound is simple: Alice can never be staler than one write interval,
because a newer point does not exist yet. Below we run the two clocks against real
Redis and measure it, instead of reading and writing in lockstep (which would show
a staleness of zero and prove nothing).

Polling faster than the writer does not help — it just returns the same point again.
We measure that too, because it is the actual cost of choosing polling.

In [ ]:
import random

random.seed(31337)   # seeded: Bob's route is reproducible

WRITE_INTERVAL_S = 5      # Bob's phone uploads a GPS point this often
DURATION_S = 60


def simulate_live_tracking(poll_interval_s):
    """
    Run Bob's writes and Alice's polls on two *independent* clocks against real
    Redis, and measure how stale Alice's view is at each poll.
    Returns (rows, max_staleness_s, redundant_polls).
    """
    lat, lon = 37.7955, -122.3935
    # Merge the two timelines and replay them in order. Writes win a tie: a poll at
    # the same instant reads the point that was just stored.
    events = ([(t, 0, 'write') for t in range(0, DURATION_S, WRITE_INTERVAL_S)] +
              [(t, 1, 'poll') for t in range(poll_interval_s, DURATION_S, poll_interval_s)])
    rows, max_stale, redundant, last_seen = [], 0.0, 0, None

    for t, _, kind in sorted(events):
        if kind == 'write':
            lat -= random.uniform(0.0003, 0.0006)
            lon += random.uniform(0.0003, 0.0006)
            r.set('live:activity:bob',
                  json.dumps({'lat': round(lat, 4), 'lon': round(lon, 4), 'sent_at': t}),
                  ex=30)
        else:
            raw = r.get('live:activity:bob')
            if raw is None:
                continue                       # nothing published yet
            seen = json.loads(raw)
            stale = t - seen['sent_at']
            max_stale = max(max_stale, stale)
            is_repeat = seen['sent_at'] == last_seen
            redundant += int(is_repeat)
            last_seen = seen['sent_at']
            rows.append((t, seen, stale, is_repeat))

    r.delete('live:activity:bob')
    return rows, max_stale, redundant


print("🏃 Bob writes every 5 s. Alice polls every 7 s.")
print()
rows, max_stale, redundant = simulate_live_tracking(poll_interval_s=7)
for t, seen, stale, is_repeat in rows:
    tag = '  (same point again)' if is_repeat else ''
    print(f"  poll t={t:>2}s → ({seen['lat']}, {seen['lon']}) "
          f"sent at t={seen['sent_at']:>2}s, {stale:.0f}s stale{tag}")

print()
print(f"  Worst staleness: {max_stale:.0f}s   Redundant polls: {redundant}/{len(rows)}")

# Alice can never be staler than one write interval -- a fresher point doesn't exist.
assert max_stale < WRITE_INTERVAL_S, (
    f"Alice saw data {max_stale:.0f}s old but Bob writes every {WRITE_INTERVAL_S}s -- "
    f"the poll is reading something other than the latest point"
)
assert redundant == 0, (
    f"polling slower than the writer should never repeat a point, got {redundant}"
)

# Now the other direction: polling FASTER than the writer buys nothing.
print()
print("🏃 Same run, but Alice polls every 2 s (faster than Bob writes):")
rows_fast, max_stale_fast, redundant_fast = simulate_live_tracking(poll_interval_s=2)
print(f"  Worst staleness: {max_stale_fast:.0f}s   "
      f"Redundant polls: {redundant_fast}/{len(rows_fast)} "
      f"({redundant_fast / len(rows_fast) * 100:.0f}% wasted)")

assert redundant_fast > 0, (
    "polling faster than the writer must return repeat points -- otherwise this "
    "cell is not demonstrating the cost of over-polling"
)
assert max_stale_fast <= max_stale, "faster polling should not make staleness worse"

print()
print("💡 Polling costs one request per client per interval whether or not anything")
print("   changed, and it can't beat the writer's interval on freshness. That's the")
print("   whole trade: no connection state, no push infrastructure, bounded lag.")
print("   Match the poll interval to the write interval — going faster only burns")
print("   requests. WebSockets win when updates are bursty or must arrive instantly.")

## 🧹 Cleanup

In [ ]:
# Clean up all Redis keys we created
r = get_redis()
for pattern in ['feed:friends:*', 'feed:fanout:*', 'live:*']:
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)
        print(f"🧹 Deleted {len(keys)} keys matching '{pattern}'")

# The pagination demo wrote 5 rows into Postgres -- take them back out.
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM activities WHERE title LIKE 'Backlog sync %'")
print(f"🧹 Deleted {cur.rowcount} seeded 'Backlog sync' activities")
conn.close()
print("🧹 Done!")

## 📚 Summary

### Key Takeaways

1. **Bi-directional friendships** use two rows per pair for simple queries
2. **Friends feed** uses a subquery + JOIN — simple but gets expensive at scale
3. **Cursor pagination** beats `LIMIT/OFFSET`, but the cursor must be *unique* —
   a bare timestamp silently drops every row tied on it
4. **Cache-aside with Redis** gives huge speedups for feed reads, and the cache key
   must contain every input the query depends on (`page_size` included)
5. **Cache invalidation** on activity completion keeps feeds fresh
6. **Fan-out-on-write** pre-builds feeds but costs more on writes
7. **Polling** (not WebSockets) is the right choice for live tracking with predictable
   updates: staleness is bounded by the *write* interval, so polling faster than the
   writer buys nothing but extra requests

### What This Toy Does *Not* Do

- The cache is invalidated, never updated. A real system would consider writing the
  new activity into the cached page instead of throwing the whole page away.
- Invalidation is best-effort: if the `DEL` fails after the Postgres write commits,
  the feed stays stale until the TTL expires. That TTL is the actual safety net.
- No privacy filtering. Real feeds respect blocked users, private activities and
  hidden start/end zones — every one of which is another predicate on the query.

### Next Up

In **Notebook 3**, we tackle **route matching and segment leaderboards** —
detecting when a GPS trace crosses a famous segment and ranking athletes
with Redis Sorted Sets.